In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================
# 0. LOAD OR GENERATE X AND y (FIX FOR ERROR)
# ==========================================
# Agar aapke paas CSV ya numpy file mein data hai, toh use yahan load karein:
# Example: 
# X = np.load("X_data.npy")
# y = np.load("y_labels.npy")

# (Agar aap testing ke liye dummy data generate karna chahte hain):
np.random.seed(42)
num_samples = 300
num_genes = 45768
X = np.random.randn(num_samples, num_genes)  # 45,768 gene features
y = np.random.randint(0, 5, size=(num_samples,))  # 5 classes (0 to 4)

# ==========================================
# 1. DEFINE CUSTOM DENSE LAYER CLASS
# ==========================================
class Dense:
    def __init__(self, input_dim, output_dim):
        # He initialization for weights (ideal for ReLU networks)
        self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        self.biases = np.zeros((1, output_dim))
        
        self.input = None
        self.dweights = None
        self.dbiases = None

    def forward(self, input_data):
        """Computes the forward pass: Z = X * W + b"""
        self.input = input_data
        return np.dot(self.input, self.weights) + self.biases

    def backward(self, output_gradient, learning_rate):
        """Computes gradients and updates parameters via gradient descent"""
        self.dweights = np.dot(self.input.T, output_gradient)
        self.dbiases = np.sum(output_gradient, axis=0, keepdims=True)
        input_gradient = np.dot(output_gradient, self.weights.T)
        
        # Parameter update
        self.weights -= learning_rate * self.dweights
        self.biases -= learning_rate * self.dbiases
        
        return input_gradient

# ==========================================
# 2. ACTIVATION & LOSS FUNCTIONS
# ==========================================
def relu(x):
    return np.maximum(0, x)

def relu_derivative(output_gradient, activated_output):
    return output_gradient * (activated_output > 0)

def softmax(x):
    exp_shifted = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_shifted / np.sum(exp_shifted, axis=1, keepdims=True)

def cross_entropy_loss(y_pred_softmax, y_true_one_hot):
    m = y_true_one_hot.shape[0]
    loss = -np.sum(y_true_one_hot * np.log(y_pred_softmax + 1e-8)) / m
    return loss

def to_one_hot(y, num_classes=5):
    """Converts integer class labels to one-hot encoded vectors for 5 classes"""
    one_hot = np.zeros((y.size, num_classes))
    one_hot[np.arange(y.size), y] = 1
    return one_hot

# ==========================================
# 3. PREPROCESSING & PCA (Using loaded X and y)
# ==========================================
print("--- Starting Preprocessing & PCA ---")
# Convert labels to one-hot encoding (5 classes: 0, 1, 2, 3, 4)
y_one_hot = to_one_hot(y, num_classes=5)

# Train-test split (80% train, 20% test)
X_train, X_test, y_train_oh, y_test_oh = train_test_split(X, y_one_hot, test_size=0.2, random_state=42)

# StandardScaler normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA Dimensionality Reduction (reducing gene features down to 64 components)
n_components = 64
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original Gene Features Shape: {X_train.shape}")
print(f"Reduced PCA Training Shape: {X_train_pca.shape}")

# ==========================================
# 4. MODEL ARCHITECTURE & TRAINING LOOP
# ==========================================
input_dim = 64     # Matches PCA components
hidden_dim = 32    # Hidden layer neurons
output_dim = 5     # 5 Cell state classes [0, 1, 2, 3, 4]

# Instantiate layers using your custom Dense class
dense1 = Dense(input_dim, hidden_dim)
dense2 = Dense(hidden_dim, output_dim)

# Hyperparameters
learning_rate = 0.05
epochs = 200

print("\n--- Training Custom Neural Network ---")
for epoch in range(epochs):
    # --- Forward Pass ---
    z1 = dense1.forward(X_train_pca)
    a1 = relu(z1)
    z2 = dense2.forward(a1)
    y_pred = softmax(z2)
    
    # --- Loss Computation ---
    loss = cross_entropy_loss(y_pred, y_train_oh)
    
    # --- Backward Pass ---
    m = X_train_pca.shape[0]
    d_z2 = (y_pred - y_train_oh) / m
    
    d_a1 = dense2.backward(d_z2, learning_rate)
    d_z1 = relu_derivative(d_a1, a1)
    _ = dense1.backward(d_z1, learning_rate)
    
    # Print progress every 40 epochs
    if (epoch + 1) % 40 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {loss:.4f}")

print("\nTraining completed successfully!")

--- Starting Preprocessing & PCA ---
Original Gene Features Shape: (240, 45768)
Reduced PCA Training Shape: (240, 64)

--- Training Custom Neural Network ---
Epoch 1/200 | Loss: 12.2127
Epoch 40/200 | Loss: 0.0071
Epoch 80/200 | Loss: 0.0034
Epoch 120/200 | Loss: 0.0024
Epoch 160/200 | Loss: 0.0018
Epoch 200/200 | Loss: 0.0015

Training completed successfully!


In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Forward Pass on Test Set ---
z1_test = dense1.forward(X_test_pca)
a1_test = np.maximum(0, z1_test) # ReLU
z2_test = dense2.forward(a1_test)
y_test_pred_probs = softmax(z2_test)

# Convert probabilities and one-hot test labels back to class integers (0 to 4)
y_pred_classes = np.argmax(y_test_pred_probs, axis=1)
y_true_classes = np.argmax(y_test_oh, axis=1)

# --- Calculate Metrics ---
test_accuracy = accuracy_score(y_true_classes, y_pred_classes)
conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
class_report = classification_report(y_true_classes, y_pred_classes)

print(f"\n--- Step 4: Model Evaluation Results ---")
print(f"Test Accuracy Score: {test_accuracy * 100:.2f}%\n")
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)


--- Step 4: Model Evaluation Results ---
Test Accuracy Score: 16.67%

Confusion Matrix:
[[1 5 2 0 2]
 [1 1 2 4 1]
 [2 1 3 6 1]
 [3 4 4 4 4]
 [3 1 1 3 1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.10      0.10      0.10        10
           1       0.08      0.11      0.10         9
           2       0.25      0.23      0.24        13
           3       0.24      0.21      0.22        19
           4       0.11      0.11      0.11         9

    accuracy                           0.17        60
   macro avg       0.16      0.15      0.15        60
weighted avg       0.17      0.17      0.17        60



In [6]:
import pandas as pd
# Save a small sample of your test features as a CSV file to test the app
pd.DataFrame(X_test[:5]).to_csv("sample_test_cell.csv", index=False)
print("Saved sample_test_cell.csv successfully!")

Saved sample_test_cell.csv successfully!


In [7]:
import pickle

# Trained components ko ek dictionary mein pack karna
model_artifacts = {
    'scaler': scaler,
    'pca': pca,
    'dense1_weights': dense1.weights,
    'dense1_biases': dense1.biases,
    'dense2_weights': dense2.weights,
    'dense2_biases': dense2.biases
}

# File mein save karna
with open('cellfate_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Model artifacts successfully saved as 'cellfate_model.pkl'!")

Model artifacts successfully saved as 'cellfate_model.pkl'!
